In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!pip install optuna
#!pip install flowio

In [ ]:

import os

TF_DETERMINISTIC_OPS = '1'
TF_CUDNN_DETERMINISTIC = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # For CuDNN ops

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import pickle
import time
import yaml
import tensorflow as tf
from tensorflow import keras

In [ ]:
#!pip install git+https://github.com/enricodidoli-source/memory-save.git
#!pip install git+https://github.com/enricodidoli-source/seed-manager.git

from memory_utils import clear_RAM, MemorySave
from seed_manager import SeedManager


In [ ]:

# = '/content/drive/MyDrive/0.Master_Thesis/'
fixed_path = 'C:/Users/admin/BinCellCNN/'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

modules_dir = f'{fixed_path}src/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)

In [ ]:

whole_exp = 'End_to_End_pipeline_try_1'


experiment_path = f'{fixed_path}experiments/experiment_{whole_exp}/'
os.makedirs(experiment_path, exist_ok=True)

config_file_path = os.path.join(experiment_path, "config.yml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

# --- Seed parameters ---
starting_seed = config['starting_seed']

# --- Environment parameters ---
colab = config['colab']
save_memory = config['save_memory']

# --- LOPOCV parameters  ---
start_lopocv_fold = config["start_lopocv_fold"]
end_lopocv_fold = config["end_lopocv_fold"]

# --- Model Parameters ---
eval_metric = config['eval_metric']

n_patients = config['n_patients']


for key, value in config.items():
    print(f"{key}: {value}")


if colab:
    mem_path = '/content/temp_data'
else:
    mem_path = f'{fixed_path}/temp_data'

mem = MemorySave(base_dir = mem_path)




In [ ]:

decache_files = ['src.shared.utils', 'src.shared.timepoints_elaboration', 'src.shared.temp_storage', 'src.shared.cv_folds',
                 'src.sample_level.bayesian_tuning', 'src.sample_level.show_results', 'src.sample_level.training',
                 'src.sample_level.testing', 'src.sample_level.pipelines', 'src.sample_level.bayesian_tuning', 'src.BinCellCNN.model']

# Remove from cache
from src.shared.utils import remove_from_cache, load_element
remove_from_cache(decache_files)

from src.shared.timepoints_elaboration import load_data, retrieve_sorted_pat_sample_ids, retrieve_samples_info, take_first_n_patients
from src.shared.temp_storage import save_test_temp
from src.shared.cv_folds import generate_LOPOCV_dicts, generate_LOPOCV_folds

from src.sample_level.bayesian_tuning import BO_tuning
from src.sample_level.show_results import show_blast_distribution_perc, show_all_LOPO_boxplots
from src.sample_level.training import fit
from src.sample_level.testing import predict
from src.sample_level.configs import get_eval_metric
from src.sample_level.pipelines import run_lopocv_fold

from src.BinCellCNN.model import BinCellCNN



# Single Cell Run

In [ ]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder_path = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

multiple_donations, all_datasets = load_data(data_path = data_folder_path, ext = extension, mem = mem)

samples_info_dict = retrieve_samples_info(data_folder_path, multiple_donations, ALL_DATASETS = None, mem = mem)
pat_sample_ids = retrieve_sorted_pat_sample_ids(samples_info_dict)


In [ ]:
tot_perc_list = show_blast_distribution_perc(ALL_DATASETS = None, multiple_donations = multiple_donations, return_perc = True, log = False, mem = mem)
print(tot_perc_list)

In [ ]:
multiple_donations = take_first_n_patients(tot_perc_list, multiple_donations, n_patients)

seed_manager = SeedManager(starting_seed=starting_seed, LOPO_fold_idx=0)
full_LOPOCV_dicts = generate_LOPOCV_dicts(multiple_donations, ALL_DATASETS= None, mem = mem)
LOPOCV_patients_folds = generate_LOPOCV_folds(full_LOPOCV_dicts,ALL_DATASETS= None, mem = mem, seed_manager = seed_manager)

In [ ]:
%%time


boxplot_data = []
thr_list = []
percs_list = []
num_samples_cum = []

for LOPOCV_fold_idx, (train_kfolds, test_pat) in enumerate(LOPOCV_patients_folds): # for each LOPO fold
  if LOPOCV_fold_idx >= start_lopocv_fold and not LOPOCV_fold_idx > end_lopocv_fold:

    test_samples_preds, original_test_y, best_thr = run_lopocv_fold(BinCellCnn = BinCellCNN,
                                                          config = config,
                                                          experiment_path = experiment_path,
                                                          LOPOCV_fold_idx = LOPOCV_fold_idx,
                                                          train_kfolds = train_kfolds,
                                                          test_pat = test_pat,
                                                          multiple_donations = multiple_donations,
                                                          ALL_DATASETS = all_datasets,
                                                          mem = mem)

    thr_list.append(best_thr)

    # === Prepare data for plot === #

    samples = np.array(multiple_donations[test_pat[0]])
    percs =  np.array(tot_perc_list)[samples]
    percs_list += list(percs)

    if len(num_samples_cum) == 0:
        num_samples_cum.append(len(percs))
    else:
        num_samples_cum.append(num_samples_cum[-1] + len(percs))

    for sample, true_y in zip(np.array(test_samples_preds).T.tolist(), original_test_y):
        print(sample)
        box_data = {"True_Label": true_y,
                    'CV_Fold_Preds': sample}

        boxplot_data.append(box_data)



results_save_dir = f'{experiment_path}/results/'
os.makedirs(results_save_dir, exist_ok=True)
plot_data = {'boxplot_data': boxplot_data, 'percs_list': percs_list,
               'thr_list': thr_list, 'num_samples_cum': num_samples_cum,
               'pat_sample_ids': pat_sample_ids}

with open(f'{results_save_dir}/plot_data.pkl', 'wb') as f:
    pickle.dump(plot_data, f)

show_all_LOPO_boxplots(boxplot_data, percs_list, thr_list, num_samples_cum,
                            save_dir = results_save_dir, file_name = 'Positivity_Score_boxplots.pdf', title = None, fold_or_seed = 'Fold', y_labels = pat_sample_ids)

mem.close()